# Four-arm evaluation — the claim that got withdrawn

Replaces `scripts/run_four_arm_eval.py`.

A reviewer pointed out the original comparison was confounded: the rules read the knowledge graph and the judge did not, so the measured variable was **evidence access**, not method. This notebook adds the missing control.

Makes ~36 Bedrock calls and takes a couple of minutes.

## Setup

Redacted readiness report — `config.py` never prints a secret.

In [1]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")          # Neo4j emits schema hints we handle
import config
print(config.report())

readiness
  neo4j      uri=neo4j+s://<redacted>.databases.neo4j.io
             user fp=6b611cde  db fp=6b611cde
             password fp=58ebef03
             configured=True
  qdrant     url=https://<redacted>.sa-east-1-0.aws.cloud.qdrant.io
             api_key fp=f3da5a0f
             configured=True
  embedding  provider=bedrock  model=amazon.titan-embed-text-v2:0  dim=1024
             configured=True
  trust      policy=v1.0  floor=0.75  min_tier=ingested


## Pre-registered design

Class balance is stated up front: 7 answers wrong, 5 correct. That is an adversarial stress-test design and **not** a prevalence estimate — a judge that says WRONG to everything scores 7/12 = 58% without discriminating at all.

In [2]:
from biomedical.eval.cases import CASES, design
import json
d = design()
print(json.dumps(d, indent=2))

{
  "case_ids": [
    "E01-consent-wrong",
    "E02-consent-right",
    "E03-catalog-wrong",
    "E04-drug-wrong",
    "E05-drug-right",
    "E06-approval-wrong",
    "E07-ae-wrong",
    "E08-ae-right",
    "E09-trial-wrong",
    "E10-trial-right",
    "E11-crossdomain-wrong",
    "E12-underpowered-right"
  ],
  "n_cases": 12,
  "class_balance": {
    "answer_wrong": 7,
    "answer_correct": 5
  },
  "balance_rationale": "adversarial stress test of a governance gate; NOT a prevalence estimate. A constant-WRONG predictor scores 7/12 here without discriminating.",
  "arms": [
    "deterministic",
    "judge_blind",
    "judge_grounded",
    "judge_rubric"
  ],
  "primary_metrics": [
    "tpr",
    "tnr",
    "kappa"
  ],
  "secondary_metrics": [
    "balanced_accuracy"
  ],
  "excluded_as_headline": [
    "accuracy"
  ],
  "falsifier": "If judge_grounded matches deterministic, the original finding was about evidence access rather than about rules versus LLMs."
}


## Parity is structural, not asserted

One `FactSet` is materialised per case and content-hashed, then handed to every arm declaring `gets_graph_facts`. Arms hold no graph client, so they *cannot* fetch anything else.

In [3]:
from biomedical.eval.arms import FactSet
fs = FactSet.of(CASES[0].facts)
print("case:", CASES[0].case_id)
print("facts supplied to every fact-receiving arm:")
print(fs.render())
print("\ncontent hash:", fs.content_hash[:24], "...")
print("order-independent:", FactSet.of(dict(reversed(list(CASES[0].facts.items())))).content_hash == fs.content_hash)

case: E01-consent-wrong
facts supplied to every fact-receiving arm:
  patient:PAT015.consent_status = Withdrawn

content hash: acfb1f82c7ebb08c174b3daf ...
order-independent: True


## Run all four arms

In [4]:
from biomedical.pipelines.evaluation import failure_table, run_four_arms

rows, per_arm, truth, computed, arms, llm = run_four_arms()
print("llm usage:", llm.usage_summary())

llm usage: 36 calls · 3936 in / 180 out tokens · $0.0145


## Per-case verdicts

`W` = the instrument said the answer is wrong.

In [5]:
hdr = f"  {'case':<22}{'truth':<8}" + "".join(f"{a.name[:15]:>16}" for a in arms)
print(hdr)
for case in CASES:
    marks = []
    for a in arms:
        r = next(x for x in rows if x["case_id"] == case.case_id and x["arm"] == a.name)
        marks.append("W" if r["says_wrong"] else ".")
    t = "WRONG" if case.answer_is_wrong else "ok"
    print(f"  {case.case_id:<22}{t:<8}" + "".join(f"{m:>16}" for m in marks))

  case                  truth      deterministic     judge_blind  judge_grounded    judge_rubric
  E01-consent-wrong     WRONG                  W               .               W               W
  E02-consent-right     ok                     .               .               .               .
  E03-catalog-wrong     WRONG                  W               W               .               W
  E04-drug-wrong        WRONG                  W               W               W               W
  E05-drug-right        ok                     .               .               .               .
  E06-approval-wrong    WRONG                  W               W               W               W
  E07-ae-wrong          WRONG                  W               W               W               W
  E08-ae-right          ok                     .               W               .               .
  E09-trial-wrong       WRONG                  W               W               W               W
  E10-trial-right       ok    

## Results

Every rate carries its denominator and a confidence interval. `accuracy` is present but is never a headline — it is determined by class balance.

In [6]:
for a in arms:
    r = computed[a.name]
    badge = f"facts={'Y' if a.gets_graph_facts else 'N'} rubric={'Y' if a.gets_rubric else 'N'}"
    tag = "   <- THE MISSING CONTROL" if a.name == "judge_grounded" else (
          "   (circularity demonstration, not a result)" if a.gets_rubric else "")
    print(f"\n{a.name}   [{badge}]{tag}")
    print(r.report())


deterministic   [facts=Y rubric=N]
  confusion        tp=7 fn=0 tn=5 fp=0  (n=12: 7 wrong / 5 correct)
  TPR (catches bad) 1.000 [0.646, 1.000] (k=7, n=7)
  TNR (passes good) 1.000 [0.566, 1.000] (k=5, n=5)
  balanced acc      1.000
  kappa             1.000 [1.000, 1.000]
  accuracy          1.000 [0.758, 1.000] (k=12, n=12)   <- determined by class balance; not a headline metric

judge_blind   [facts=N rubric=N]
  confusion        tp=6 fn=1 tn=3 fp=2  (n=12: 7 wrong / 5 correct)
  TPR (catches bad) 0.857 [0.487, 0.974] (k=6, n=7)
  TNR (passes good) 0.600 [0.231, 0.882] (k=3, n=5)
  balanced acc      0.729
  kappa             0.471 [-0.125, 1.000]
  accuracy          0.750 [0.468, 0.911] (k=9, n=12)   <- determined by class balance; not a headline metric

judge_grounded   [facts=Y rubric=N]   <- THE MISSING CONTROL
  confusion        tp=6 fn=1 tn=5 fp=0  (n=12: 7 wrong / 5 correct)
  TPR (catches bad) 0.857 [0.487, 0.974] (k=6, n=7)
  TNR (passes good) 1.000 [0.566, 1.000] (k=5, n=5

## Reading

In [7]:
blind, ground, det = computed["judge_blind"], computed["judge_grounded"], computed["deterministic"]
print(f"  judge_blind     TNR {blind.tnr}   kappa {blind.kappa:.3f} {blind.kappa_ci}")
print(f"  judge_grounded  TNR {ground.tnr}   kappa {ground.kappa:.3f} {ground.kappa_ci}")
print(f"  deterministic   TNR {det.tnr}   kappa {det.kappa:.3f} {det.kappa_ci}")
print()
if ground.kappa > blind.kappa + 0.1:
    print("  Giving the judge the SAME facts materially improved it.")
    print("  The original finding was substantially about EVIDENCE ACCESS,")
    print("  not about rules versus LLMs. The reviewer was right; the")
    print("  confound was real and the original claim does not survive it.")
print()
print("  Note the blind arm's kappa interval includes zero, so even its point")
print("  estimate is not distinguishable from no skill at n=12.")

  judge_blind     TNR 0.600 [0.231, 0.882] (k=3, n=5)   kappa 0.471 [-0.125, 1.000]
  judge_grounded  TNR 1.000 [0.566, 1.000] (k=5, n=5)   kappa 0.833 [0.437, 1.000]
  deterministic   TNR 1.000 [0.566, 1.000] (k=5, n=5)   kappa 1.000 [1.000, 1.000]

  Giving the judge the SAME facts materially improved it.
  The original finding was substantially about EVIDENCE ACCESS,
  not about rules versus LLMs. The reviewer was right; the
  confound was real and the original claim does not survive it.

  Note the blind arm's kappa interval includes zero, so even its point
  estimate is not distinguishable from no skill at n=12.


## Failure analysis

By arm and by direction — including the deterministic arm's own errors.

In [8]:
for arm, info in failure_table(rows).items():
    print(f"  {arm:<18} {info['errors']} error(s)")
    if info["errors"]:
        print(f"                     {info['kinds']}")
        print(f"                     {info['categories']}")

  judge_grounded     1 error(s)
                     {'false_pass (missed a bad answer)': 1}
                     {'Aggregate Governance': 1}
  judge_rubric       0 error(s)
  judge_blind        3 error(s)
                     {'false_pass (missed a bad answer)': 1, 'false_fail (failed a good answer)': 2}
                     {'Consent Governance': 1, 'Adverse Event Safety': 1, 'Trial Status': 1}
  deterministic      0 error(s)


## Honest limits

- **n = 12**, five true-negative opportunities. Every interval is wide.
- One domain, one judge model, one prompt style, temperature 0. No claim about LLM judges in general.
- The deterministic arm was **built against these cases** — it encodes the contrasts they test. Read its score as "a correctly specified rule set catches what it was specified for", not as generality. v1's deterministic arm scored 0.917 on the same design and missed one case; that is the more realistic figure.
- Cases are authored and facts are synthetic.

In [9]:
from biomedical.foundation import RunManifest
m = RunManifest.create(
    design=d,
    source_versions={"graph_facts": "authored-v1", "judge_model": "us.anthropic.claude-sonnet-4-6"},
    slice_params={"suite": "governance-module-E"}, seeds={"bootstrap": 20261011},
    trust_policy_version=config.load().trust.policy_version)
print(m.to_json())

{
  "code_commit": "c1fda54",
  "created_at": "2026-10-11T13:09:22.469996+00:00",
  "design_hash": "28a95d9341f607bf",
  "notes": {},
  "python_version": "3.12.3",
  "run_id": "run_20261011_130922_16f7ef",
  "seeds": {
    "bootstrap": 20261011
  },
  "slice_params": {
    "suite": "governance-module-E"
  },
  "source_versions": {
    "graph_facts": "authored-v1",
    "judge_model": "us.anthropic.claude-sonnet-4-6"
  },
  "trust_policy_version": "1.0"
}


## Run-to-run variance — found by re-running this notebook

The numbers above are **not** identical to the ones in `FINDINGS.md`, which came
from an earlier run of the same design at the same temperature (0):

| | `FINDINGS.md` run | this run |
|---|---|---|
| judge_blind TNR | 0.400 (2/5) | 0.600 (3/5) |
| judge_blind kappa | 0.273 [-0.286, 0.800] | 0.471 [-0.125, 1.000] |
| judge_grounded kappa | 0.833 [0.437, 1.000] | 0.833 [0.437, 1.000] |
| deterministic kappa | 1.000 | 1.000 |

**The judge is not deterministic across runs even at temperature 0.** The
deterministic and grounded arms reproduced exactly; the blind arm moved by one
case, which at n=5 true negatives shifts TNR by 20 points.

Two things follow, and both cut against the headline:

1. **A single run's point estimate is not stable.** Quoting "kappa 0.273" as a
   property of the judge was already unsupportable at this n; it turns out not to
   be reproducible either.
2. **The direction survives but is not strongly established.** Grounded (0.833)
   still exceeds blind (0.471), but the two kappa intervals overlap heavily
   ([-0.125, 1.000] vs [0.437, 1.000]). The confound is real and the original
   claim is still withdrawn — but the replacement claim needs more runs before
   its magnitude means anything.

The honest report is therefore: *giving the judge the same facts improved it in
both runs, by an amount this design cannot yet pin down.* The fix is repeated
runs with the variance reported, not a larger adjective.